# Sales Data Wrangling and Visualizations

Explore the `sales_items.csv` dataset by inspecting and cleaning records, deriving profit and shipping duration, summarizing sales, and comparing revenue over time.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

## 1. Import Libraries and Load the CSV

Load the provided CSV. Read order IDs as strings so missing IDs do not cause the column to be treated as numeric.

In [ ]:
data_path = Path("/workspaces/codespaces-jupyter/Python &amp; csv/sales_items.csv")
df = pd.read_csv(data_path, dtype={"Order ID": "string"})

print(f"Rows: {len(df):,} | Columns: {len(df.columns)}")
df.head()

## 2. Inspect and Clean the Data

Review data types, missing values, and exact duplicate rows. Trim whitespace from text columns and replace missing categorical values with `"Unknown"`. Keep missing dates and order IDs missing, and leave missing numeric values as `NaN` rather than inventing values. Drop exact duplicate records.

In [ ]:
print("Data types:")
display(df.dtypes)

print("Missing values before cleaning:")
display(df.isna().sum().sort_values(ascending=False))

duplicate_count = df.duplicated().sum()
print(f"Exact duplicate rows: {duplicate_count}")

text_columns = df.select_dtypes(include=["object", "string"]).columns
for column in text_columns:
    df[column] = df[column].str.strip()

categorical_columns = [
    "Region",
    "Country",
    "Item Type",
    "Sales Channel",
    "Order Priority",
]
for column in categorical_columns:
    df[column] = df[column].fillna("Unknown")
    df.loc[df[column].eq(""), column] = "Unknown"

df = df.drop_duplicates().copy()

print("Missing values after cleaning:")
display(df.isna().sum().sort_values(ascending=False))
df.head()

## 3. Parse Dates and Add Derived Columns

Parse dates with invalid or missing values coerced to `NaT`. Calculate profit as revenue minus cost. Shipping duration is calculated in days only when both dates are available.

In [ ]:
for column in ["Order Date", "Ship Date"]:
    df[column] = pd.to_datetime(df[column], errors="coerce")

numeric_columns = [
    "Units Sold",
    "Unit Price",
    "Unit Cost",
    "Total Revenue",
    "Total Cost",
]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df["Profit"] = df["Total Revenue"] - df["Total Cost"]
df["Shipping Duration Days"] = (df["Ship Date"] - df["Order Date"]).dt.days

print("Date values that could not be parsed:")
display(df[["Order Date", "Ship Date"]].isna().sum())

df[["Order Date", "Ship Date", "Total Revenue", "Total Cost", "Profit", "Shipping Duration Days"]].head()

## 4. Summarize Sales by Region and Item Type

Aggregate revenue, cost, profit, and units sold by region and item type. Also create separate region and item type totals for easier chart comparisons.

In [ ]:
metrics = {
    "Total Revenue": "sum",
    "Total Cost": "sum",
    "Profit": "sum",
    "Units Sold": "sum",
}

sales_by_region_item = (
    df.groupby(["Region", "Item Type"], dropna=False)
    .agg(metrics)
    .sort_values("Total Revenue", ascending=False)
)

sales_by_region = (
    df.groupby("Region", dropna=False)
    .agg(metrics)
    .sort_values("Total Revenue", ascending=False)
)

sales_by_item = (
    df.groupby("Item Type", dropna=False)
    .agg(metrics)
    .sort_values("Total Revenue", ascending=False)
)

display(sales_by_region_item)

## 5. Visualize Revenue and Profit

Compare total revenue and profit across regions and item types using labeled bar charts.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

sales_by_region["Total Revenue"].sort_values().plot(
    kind="barh", ax=axes[0, 0], color="steelblue"
)
axes[0, 0].set_title("Revenue by Region")
axes[0, 0].set_xlabel("Revenue")
axes[0, 0].set_ylabel("Region")

sales_by_region["Profit"].sort_values().plot(
    kind="barh", ax=axes[0, 1], color="seagreen"
)
axes[0, 1].set_title("Profit by Region")
axes[0, 1].set_xlabel("Profit")
axes[0, 1].set_ylabel("Region")

sales_by_item["Total Revenue"].sort_values().plot(
    kind="barh", ax=axes[1, 0], color="steelblue"
)
axes[1, 0].set_title("Revenue by Item Type")
axes[1, 0].set_xlabel("Revenue")
axes[1, 0].set_ylabel("Item Type")

sales_by_item["Profit"].sort_values().plot(
    kind="barh", ax=axes[1, 1], color="seagreen"
)
axes[1, 1].set_title("Profit by Item Type")
axes[1, 1].set_xlabel("Profit")
axes[1, 1].set_ylabel("Item Type")

fig.tight_layout()
plt.show()

## 6. Compare Sales Channels Over Time

Aggregate revenue by order month and sales channel. Rows without a valid order date are excluded from this time-series analysis.

In [ ]:
dated_sales = df.dropna(subset=["Order Date"]).copy()
dated_sales["Order Month"] = dated_sales["Order Date"].dt.to_period("M").dt.to_timestamp()

monthly_channel_sales = (
    dated_sales.groupby(["Order Month", "Sales Channel"])["Total Revenue"]
    .sum()
    .unstack(fill_value=0)
    .sort_index()
)

display(monthly_channel_sales.head())

ax = monthly_channel_sales.plot(figsize=(14, 6), marker="o")
ax.set_title("Monthly Revenue by Sales Channel")
ax.set_xlabel("Order Month")
ax.set_ylabel("Revenue")
ax.legend(title="Sales Channel")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()